# Fase II · Verificación de la integridad y consistencia de los registros

Se comprueba la integridad y completitud de los registros, así como el formato y la consistencia de las señales vibracionales.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW = Path.cwd().parent / "data" / "raw"
INTERIM = RAW.parent / "interim"
INTERIM.mkdir(exist_ok=True)

EXPERIMENTOS = {
    "Experimento 1": RAW / "1st_test" / "1st_test",
    "Experimento 2": RAW / "2nd_test" / "2nd_test",
    "Experimento 3": RAW / "3rd_test" / "4th_test" / "txt",
}

## 1. Revisión del posible duplicado

Se comparan las dos capturas tomadas con solo 13 segundos de diferencia, el único caso realmente sospechoso de duplicado, verificando si su contenido es idéntico y comparando su RMS por canal.

In [2]:
import numpy as np

c1 = EXPERIMENTOS["Experimento 1"]
a = pd.read_csv(c1 / "2003.11.23.20.01.11", sep="\t", header=None)
b = pd.read_csv(c1 / "2003.11.23.20.01.24", sep="\t", header=None)

print("¿Contenido idéntico?", a.equals(b))
pd.DataFrame({
    "RMS anterior": np.sqrt((a**2).mean()),
    "RMS siguiente": np.sqrt((b**2).mean()),
}).round(4)

¿Contenido idéntico? False


,RMS anterior,RMS siguiente
0,0.1557,0.1547
1,0.1481,0.1487
2,0.1516,0.1507
3,0.1420,0.1412
4,0.1665,0.1650
5,0.1715,0.1694
6,0.1743,0.1738
7,0.1758,0.1772


Las capturas no son idénticas y sus RMS difieren alrededor de un 1 % en todos los canales. Son dos capturas legítimas tomadas muy seguidas al reanudar el registro, por lo que se conservan ambas.

## 2. Verificación de filas, canales y valores faltantes

Se leen los 9.464 archivos para comprobar que todos tengan 20.480 muestras, el número correcto de canales y ningún valor faltante, porque un archivo incompleto o corrupto distorsionaría cualquier indicador. Como leer todo es la operación más lenta, en la misma lectura se calculan RMS, pico y curtosis por canal, y el resultado se guarda en `data/interim/resumen_por_archivo.csv` para no repetirla. Esta celda tarda varios minutos; si el archivo ya existe, no es necesario volver a ejecutarla.

In [3]:
registros = []

for nombre, carpeta in EXPERIMENTOS.items():
    archivos_exp = sorted(f for f in carpeta.iterdir() if f.is_file())
    for i, f in enumerate(archivos_exp, start=1):
        datos = pd.read_csv(f, sep="\t", header=None)

        fila = {
            "experimento": nombre,
            "archivo": f.name,
            "fecha": pd.to_datetime(f.name, format="%Y.%m.%d.%H.%M.%S"),
            "filas": datos.shape[0],
            "columnas": datos.shape[1],
            "faltantes": int(datos.isna().sum().sum()),
        }
        for c in datos.columns:
            x = datos[c]
            fila[f"rms_{c}"] = np.sqrt((x**2).mean())
            fila[f"pico_{c}"] = x.abs().max()
            fila[f"curtosis_{c}"] = x.kurt()
        registros.append(fila)

        if i % 500 == 0 or i == len(archivos_exp):
            print(f"{nombre}: {i}/{len(archivos_exp)} archivos procesados")

resumen = pd.DataFrame(registros)

destino = RAW.parent / "interim"
destino.mkdir(exist_ok=True)
resumen.to_csv(destino / "resumen_por_archivo.csv", index=False)
print("\nGuardado en:", destino / "resumen_por_archivo.csv")

Experimento 1: 500/2156 archivos procesados
Experimento 1: 1000/2156 archivos procesados
Experimento 1: 1500/2156 archivos procesados
Experimento 1: 2000/2156 archivos procesados
Experimento 1: 2156/2156 archivos procesados
Experimento 2: 500/984 archivos procesados
Experimento 2: 984/984 archivos procesados
Experimento 3: 500/6324 archivos procesados
Experimento 3: 1000/6324 archivos procesados
Experimento 3: 1500/6324 archivos procesados
Experimento 3: 2000/6324 archivos procesados
Experimento 3: 2500/6324 archivos procesados
Experimento 3: 3000/6324 archivos procesados
Experimento 3: 3500/6324 archivos procesados
Experimento 3: 4000/6324 archivos procesados
Experimento 3: 4500/6324 archivos procesados
Experimento 3: 5000/6324 archivos procesados
Experimento 3: 5500/6324 archivos procesados
Experimento 3: 6000/6324 archivos procesados
Experimento 3: 6324/6324 archivos procesados

Guardado en: C:\Users\rayo1\Documents\eda proyecto\bearing-predictive-maintenance\data\interim\resumen_po

In [4]:
resumen.groupby("experimento")[["filas", "columnas", "faltantes"]].agg(["min", "max"])

filas        columnas     faltantes    
                 min    max      min max       min max
experimento                                           
Experimento 1  20480  20480        8   8         0   0
Experimento 2  20480  20480        4   4         0   0
Experimento 3  20480  20480        4   4         0   0

El dataset es íntegro en contenido: las 9.464 capturas tienen exactamente 20.480 muestras, el experimento 1 tiene 8 canales en todos sus archivos y los otros dos tienen 4, y no hay un solo valor faltante ni archivos corruptos. Las únicas irregularidades del dataset son temporales.

## 3. Consistencia de las señales: saturación, señales planas y valores extremos

Se buscan señales casi planas (sensor desconectado o máquina detenida), saturación (señal recortada en el límite del sensor) y valores fuera del rango típico de cada canal. Una captura se considera casi plana si su desviación estándar es menor al 20 % de la típica de su canal, y hay posible saturación si 10 o más muestras repiten exactamente el valor máximo o mínimo; se usa la referencia de cada canal porque cada sensor tiene su propio nivel. La primera celda vuelve a leer los archivos (varios minutos).

In [5]:
registros_c = []

for nombre, carpeta in EXPERIMENTOS.items():
    archivos_exp = sorted(f for f in carpeta.iterdir() if f.is_file())
    for i, f in enumerate(archivos_exp, start=1):
        x = pd.read_csv(f, sep="\t", header=None).values
        fila = {"experimento": nombre, "archivo": f.name,
                "fecha": pd.to_datetime(f.name, format="%Y.%m.%d.%H.%M.%S")}
        for c in range(x.shape[1]):
            s = x[:, c]
            fila[f"min_{c}"] = s.min()
            fila[f"max_{c}"] = s.max()
            fila[f"std_{c}"] = s.std()
            fila[f"en_max_{c}"] = int((s == s.max()).sum())
            fila[f"en_min_{c}"] = int((s == s.min()).sum())
        registros_c.append(fila)
        if i % 500 == 0 or i == len(archivos_exp):
            print(f"{nombre}: {i}/{len(archivos_exp)}")

consistencia = pd.DataFrame(registros_c)
consistencia.to_csv(INTERIM / "consistencia_por_archivo.csv", index=False)
print("Guardado.")

Experimento 1: 500/2156
Experimento 1: 1000/2156
Experimento 1: 1500/2156
Experimento 1: 2000/2156
Experimento 1: 2156/2156
Experimento 2: 500/984
Experimento 2: 984/984
Experimento 3: 500/6324
Experimento 3: 1000/6324
Experimento 3: 1500/6324
Experimento 3: 2000/6324
Experimento 3: 2500/6324
Experimento 3: 3000/6324
Experimento 3: 3500/6324
Experimento 3: 4000/6324
Experimento 3: 4500/6324
Experimento 3: 5000/6324
Experimento 3: 5500/6324
Experimento 3: 6000/6324
Experimento 3: 6324/6324
Guardado.


In [6]:
# Reorganizamos la tabla: una fila por captura y canal
partes = []
for c in range(8):
    cols = [f"min_{c}", f"max_{c}", f"std_{c}", f"en_max_{c}", f"en_min_{c}"]
    if cols[0] not in consistencia:
        continue
    tmp = consistencia[["experimento", "archivo", "fecha"] + cols].dropna()
    tmp.columns = ["experimento", "archivo", "fecha", "min", "max", "std", "en_max", "en_min"]
    tmp["canal"] = c
    partes.append(tmp)
largo = pd.concat(partes, ignore_index=True)

# Desviación de cada captura relativa a la típica de su canal
largo["std_rel"] = largo["std"] / largo.groupby(["experimento", "canal"])["std"].transform("median")

print("Rango global de valores por experimento y canal:")
display(largo.groupby(["experimento", "canal"])[["min", "max"]].agg(["min", "max"]).round(3))

planas = largo[largo["std_rel"] < 0.2]
print(f"\nSeñales casi planas (variación menor al 20 % de la típica del canal): {len(planas)}")
display(planas.sort_values("fecha"))

saturadas = largo[(largo["en_max"] >= 10) | (largo["en_min"] >= 10)]
print(f"\nPosible saturación (10 o más muestras repitiendo el valor extremo): {len(saturadas)}")
display(saturadas.sort_values("fecha").head(20))

Rango global de valores por experimento y canal:


min           max       
                       min    max    min    max
experimento   canal                            
Experimento 1 0     -1.042 -0.510  0.278  0.701
              1     -1.086 -0.437  0.293  1.025
              2     -0.991 -0.471  0.215  0.728
              3     -0.830 -0.420  0.164  0.552
              4     -5.000 -0.454  0.273  4.998
              5     -5.000 -0.508  0.295  4.724
              6     -3.975 -0.308  0.159  3.938
              7     -3.801 -0.430  0.222  3.596
Experimento 2 0     -5.000  0.000  0.002  4.998
              1     -0.884  0.000  0.002  1.118
              2     -1.035  0.000  0.002  1.284
              3     -0.566 -0.002  0.002  0.706
Experimento 3 0     -1.072 -0.002  0.005  0.964
              1     -1.292  0.000  0.007  0.962
              2     -5.000  0.000  0.007  4.998
              3     -1.104 -0.002  0.005  1.082


Señales casi planas (variación menor al 20 % de la típica del canal): 12


,experimento,archivo,fecha,min,max,std,en_max,en_min,canal,std_rel
3138,Experimento 2,2004.02.19.06.12.39,2004-02-19 06:12:39,0.000,0.005,0.000987,968.0,2918.0,0,0.012538
12602,Experimento 2,2004.02.19.06.12.39,2004-02-19 06:12:39,0.000,0.007,0.001490,45.0,1.0,1,0.015737
22066,Experimento 2,2004.02.19.06.12.39,2004-02-19 06:12:39,0.000,0.007,0.001496,26.0,1.0,2,0.014817
31530,Experimento 2,2004.02.19.06.12.39,2004-02-19 06:12:39,-0.002,0.005,0.001204,1456.0,2.0,3,0.021504
3139,Experimento 2,2004.02.19.06.22.39,2004-02-19 06:22:39,-0.005,0.002,0.001000,33.0,17.0,0,0.012694
12603,Experimento 2,2004.02.19.06.22.39,2004-02-19 06:22:39,-0.002,0.002,0.001021,338.0,7518.0,1,0.010788
22067,Experimento 2,2004.02.19.06.22.39,2004-02-19 06:22:39,-0.005,0.002,0.000996,269.0,1.0,2,0.009866
31531,Experimento 2,2004.02.19.06.22.39,2004-02-19 06:22:39,-0.005,0.002,0.001280,17.0,1522.0,3,0.022855
9463,Experimento 3,2004.04.18.02.42.55,2004-04-18 02:42:55,-0.002,0.005,0.001143,805.0,1.0,0,0.015625
18927,Experimento 3,2004.04.18.02.42.55,2004-04-18 02:42:55,0.000,0.007,0.001351,20.0,1407.0,1,0.017185



Posible saturación (10 o más muestras repitiendo el valor extremo): 17


,experimento,archivo,fecha,min,max,std,en_max,en_min,canal,std_rel
3131,Experimento 2,2004.02.19.05.02.39,2004-02-19 05:02:39,-5.000,4.998,0.672129,1.0,19.0,0,8.535500
3135,Experimento 2,2004.02.19.05.42.39,2004-02-19 05:42:39,-5.000,4.998,0.724996,5.0,17.0,0,9.206873
31530,Experimento 2,2004.02.19.06.12.39,2004-02-19 06:12:39,-0.002,0.005,0.001204,1456.0,2.0,3,0.021504
12602,Experimento 2,2004.02.19.06.12.39,2004-02-19 06:12:39,0.000,0.007,0.001490,45.0,1.0,1,0.015737
22066,Experimento 2,2004.02.19.06.12.39,2004-02-19 06:12:39,0.000,0.007,0.001496,26.0,1.0,2,0.014817
3138,Experimento 2,2004.02.19.06.12.39,2004-02-19 06:12:39,0.000,0.005,0.000987,968.0,2918.0,0,0.012538
3139,Experimento 2,2004.02.19.06.22.39,2004-02-19 06:22:39,-0.005,0.002,0.001000,33.0,17.0,0,0.012694
12603,Experimento 2,2004.02.19.06.22.39,2004-02-19 06:22:39,-0.002,0.002,0.001021,338.0,7518.0,1,0.010788
31531,Experimento 2,2004.02.19.06.22.39,2004-02-19 06:22:39,-0.005,0.002,0.001280,17.0,1522.0,3,0.022855
22067,Experimento 2,2004.02.19.06.22.39,2004-02-19 06:22:39,-0.005,0.002,0.000996,269.0,1.0,2,0.009866


Las señales son consistentes durante casi toda la vida de los rodamientos; las anomalías se concentran en la última hora de los experimentos 2 y 3 y tienen una explicación física clara.

- **Rango del sensor:** todos los valores están entre −5 y +5, que corresponde al límite de medición del sistema de adquisición. No hay valores físicamente imposibles.
- **Señales planas:** solo 3 capturas, en sus 4 canales a la vez: las dos últimas del experimento 2 (19 de febrero, 06:12:39 y 06:22:39) y la última del experimento 3 (18 de abril, 02:42:55). Su variación es apenas el 1 a 2 % de la típica, lo que indica que la máquina ya estaba detenida. No son representativas y deben excluirse. El experimento 1 no tiene capturas planas.
- **Saturación real:** ocurre únicamente en los rodamientos que fallaron y en los últimos minutos de la prueba: en el canal 0 del experimento 2 (05:02:39 y 05:42:39, hasta 19 muestras recortadas en −5) y en el canal 2 del experimento 3 (01:32:55, 02:22:55 y 02:32:55, hasta 43 muestras en −5). En el experimento 1, los canales del rodamiento 3 también tocan el límite, pero con menos de 10 muestras por captura.
- **Falsos positivos del criterio de saturación:** de las 17 detecciones, 12 corresponden a las capturas planas; en ellas el máximo y el mínimo se repiten porque la señal casi no varía y está redondeada a tres decimales, no porque esté recortada.

**Decisiones para la limpieza:** excluir las 3 capturas planas; conservar las capturas saturadas, porque representan la falla, pero marcarlas, ya que el recorte subestima su RMS y distorsiona su curtosis.